# 8.2 端侧训练优化

端侧设备内存有限、算力有限，需要特殊的训练优化策略才能在端侧完成模型微调。核心技术包括选择性反向传播、梯度累积、混合精度训练和内存高效优化器。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 选择性反向传播（Selective Backpropagation）

仅对部分层进行反向传播，冻结其余层，减少训练的计算量和内存占用。

In [ ]:
class SelectiveBackpropModel(nn.Module):
    """支持选择性反向传播的模型"""
    def __init__(self, dim=64, n_layers=6, n_classes=10, trainable_layers=None):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'linear': nn.Linear(dim, dim),
                'act': nn.ReLU(),
            }) for _ in range(n_layers)
        ])
        self.head = nn.Linear(dim, n_classes)
        self.n_layers = n_layers
        self.trainable_layers = trainable_layers or [n_layers - 1]
        self._freeze_layers()

    def _freeze_layers(self):
        for i in range(self.n_layers):
            if i not in self.trainable_layers:
                for p in self.layers[i].parameters():
                    p.requires_grad = False

    def forward(self, x):
        for layer in self.layers:
            x = layer['act'](layer['linear'](x))
        return self.head(x)

dim, n_classes = 64, 10
x_data = torch.randn(256, dim)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

configs = [
    ('全层训练', list(range(6))),
    ('仅最后2层', [4, 5]),
    ('仅最后1层', [5]),
]

print(f"=== 选择性反向传播 ===")
print(f"\n{'配置':<15} {'可训练参数':<15} {'训练时间':<12} {'准确率':<10}")
print("-" * 52)

for name, trainable in configs:
    model = SelectiveBackpropModel(dim, 6, n_classes, trainable)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    start = time.time()
    for epoch in range(10):
        for x, y in loader:
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    train_time = time.time() - start

    with torch.no_grad():
        acc = (model(x_data).argmax(1) == y_data).float().mean()

    print(f"{name:<15} {trainable_params:<15,} {train_time:<12.4f} {acc:<10.4f}")

print(f"\n选择性反向传播: 冻结大部分层，仅训练关键层")
print(f"优势: 大幅减少梯度计算和优化器状态的内存占用")

### 梯度累积 & 混合精度训练

In [ ]:
class GradientAccumulator:
    """梯度累积器: 小batch累积模拟大batch"""
    def __init__(self, model, optimizer, accumulation_steps=4):
        self.model = model
        self.optimizer = optimizer
        self.accumulation_steps = accumulation_steps
        self.step_count = 0

    def step(self, loss):
        scaled_loss = loss / self.accumulation_steps
        scaled_loss.backward()
        self.step_count += 1
        if self.step_count % self.accumulation_steps == 0:
            self.optimizer.step()
            self.optimizer.zero_grad()
            return True
        return False

class MixedPrecisionTrainer:
    """混合精度训练器"""
    def __init__(self, model, optimizer, scaler=None):
        self.model = model
        self.optimizer = optimizer
        self.scaler = scaler

    def train_step(self, x, y):
        self.model.train()
        with torch.autocast(device_type='cpu', dtype=torch.float16, enabled=True):
            output = self.model(x)
            loss = F.cross_entropy(output, y)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(256, 64)
y = torch.randint(0, 10, (256,))
small_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x, y), batch_size=16)

accumulator = GradientAccumulator(model, optimizer, accumulation_steps=4)
losses = []
for x_batch, y_batch in small_loader:
    loss = F.cross_entropy(model(x_batch), y_batch)
    accumulator.step(loss)
    losses.append(loss.item())

print(f"=== 梯度累积 ===")
print(f"小batch大小: 16")
print(f"累积步数: 4")
print(f"等效batch大小: {16 * 4}")
print(f"损失: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"\n梯度累积优势:")
print(f"1. 小batch内存友好: 每次仅计算16样本的激活")
print(f"2. 等效大batch效果: 累积4步后更新，等效batch=64")
print(f"3. 端侧必备: 端侧内存有限，无法使用大batch")

print(f"\n=== 混合精度训练 ===")
print(f"FP32权重 + FP16前向 → 减少激活内存")
print(f"FP16梯度 → 减少梯度内存")
print(f"FP32优化器更新 → 保持数值稳定性")
print(f"\n内存节省: 激活和梯度减半，优化器状态仍FP32")

### 内存高效优化器

In [ ]:
class Adam8bit:
    """8-bit Adam优化器：将优化器状态（exp_avg, exp_avg_sq）量化为INT8存储，降低内存占用"""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.betas = betas
        self.eps = eps
        self.state = {}
        for p in self.params:
            exp_avg = torch.zeros_like(p.data)
            exp_avg_sq = torch.zeros_like(p.data)
            q_m, s_m = self._quantize_8bit(exp_avg)
            q_v, s_v = self._quantize_8bit(exp_avg_sq)
            self.state[p] = {
                'step': 0,
                'exp_avg_q': q_m,
                'exp_avg_scale': s_m,
                'exp_avg_sq_q': q_v,
                'exp_avg_sq_scale': s_v,
            }

    def _quantize_8bit(self, tensor):
        scale = tensor.abs().max() / 127.0
        scale = torch.clamp(scale, min=1e-8)
        q = torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
        return q, scale

    def _dequantize_8bit(self, q, scale):
        return q.float() * scale

    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            state = self.state[p]
            state['step'] += 1
            beta1, beta2 = self.betas

            exp_avg = self._dequantize_8bit(state['exp_avg_q'], state['exp_avg_scale'])
            exp_avg_sq = self._dequantize_8bit(state['exp_avg_sq_q'], state['exp_avg_sq_scale'])

            exp_avg = beta1 * exp_avg + (1 - beta1) * p.grad
            exp_avg_sq = beta2 * exp_avg_sq + (1 - beta2) * p.grad ** 2

            state['exp_avg_q'], state['exp_avg_scale'] = self._quantize_8bit(exp_avg)
            state['exp_avg_sq_q'], state['exp_avg_sq_scale'] = self._quantize_8bit(exp_avg_sq)

            bias_correction1 = 1 - beta1 ** state['step']
            bias_correction2 = 1 - beta2 ** state['step']

            step_size = self.lr / bias_correction1
            denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add(self.eps)
            p.data -= step_size * exp_avg / denom

import math

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))

adam32_params = sum(p.numel() * 4 * 2 for p in model.parameters())
adam8_params = sum(p.numel() * 1 * 2 for p in model.parameters())

print(f"=== 优化器内存对比 ===")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"\n优化器状态内存 (m + v 两个状态):")
print(f"  Adam (FP32): {adam32_params/1024:.2f} KB")
print(f"  8-bit Adam: {adam8_params/1024:.2f} KB")
print(f"  节省: {(1-adam8_params/adam32_params)*100:.0f}%")

print(f"\n=== 端侧训练优化总结 ===")
print(f"\n{'技术':<20} {'内存节省':<15} {'适用场景':<25}")
print("-" * 60)
techniques = [
    ('选择性反向传播', '60-80%', '仅微调最后几层'),
    ('梯度累积', '50-75%', '小batch模拟大batch'),
    ('混合精度(FP16)', '30-50%', '通用训练加速'),
    ('8-bit优化器', '50-75%', '优化器状态压缩'),
    ('QLoRA', '80-90%', '量化+LoRA微调'),
]
for name, saving, scene in techniques:
    print(f"{name:<20} {saving:<15} {scene:<25}")

print(f"\n端侧训练最佳实践: QLoRA + 梯度累积 + 8-bit优化器")
print(f"可将7B模型的微调显存需求从28GB降至<6GB")

### 产业级实战：真实混合精度训练 + bitsandbytes 8-bit Adam

生产环境中，混合精度训练使用 PyTorch 原生 AMP (Automatic Mixed Precision)，8-bit 优化器使用 bitsandbytes。以下展示真实的端到端训练流程。

In [ ]:
import bitsandbytes as bnb
import time

class SimpleClassifier(nn.Module):
    def __init__(self, dim=128, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

def train_with_config(model, optimizer_cls, lr, use_amp, n_steps=100, device='cpu'):
    model = model.to(device)
    model.train()
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and torch.cuda.is_available())

    total_loss = 0
    start = time.perf_counter()
    for step in range(n_steps):
        x = torch.randn(32, 128, device=device)
        y = torch.randint(0, 10, (32,), device=device)

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp and torch.cuda.is_available()):
            output = model(x)
            loss = F.cross_entropy(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        total_loss += loss.item()

    elapsed = time.perf_counter() - start
    return total_loss / n_steps, elapsed

device = 'cuda' if torch.cuda.is_available() else 'cpu'

configs = [
    ('FP32 + AdamW', torch.optim.AdamW, 1e-3, False),
    ('AMP FP16 + AdamW', torch.optim.AdamW, 1e-3, True),
    ('FP32 + 8-bit Adam', lambda p, lr: bnb.optim.Adam8bit(p, lr=lr), 1e-3, False),
    ('AMP FP16 + 8-bit Adam', lambda p, lr: bnb.optim.Adam8bit(p, lr=lr), 1e-3, True),
]

print(f"=== 产业级训练优化对比 ===")
print(f"设备: {device}")
print(f"\n{'配置':<28} {'Loss':<10} {'时间(s)':<10} {'优化器内存':<15}")
print("-" * 63)

for name, opt_cls, lr, use_amp in configs:
    model = SimpleClassifier()
    avg_loss, elapsed = train_with_config(model, opt_cls, lr, use_amp, n_steps=50, device=device)

    opt = opt_cls(model.parameters(), lr=lr)
    opt_mem = 0
    for group in opt.param_groups:
        for p in group['params']:
            if p in opt.state:
                for v in opt.state[p].values():
                    if isinstance(v, torch.Tensor):
                        opt_mem += v.numel() * v.element_size()
    opt_mem_mb = opt_mem / 1024 / 1024

    print(f"{name:<28} {avg_loss:<10.4f} {elapsed:<10.2f} {opt_mem_mb:<15.2f} MB")
    del model, opt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n产业界端侧训练最佳实践:")
print(f"1. 混合精度: torch.autocast + GradScaler (GPU上2x加速+50%内存节省)")
print(f"2. 8-bit优化器: bitsandbytes.optim.Adam8bit (优化器内存节省75%)")
print(f"3. 梯度检查点: torch.utils.checkpoint (激活内存节省60%+)")
print(f"4. 梯度累积: 小batch累积模拟大batch (端侧内存受限时)")
print(f"5. QLoRA: NF4基座 + LoRA微调 (7B模型<6GB显存可训练)")